In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# ── Load features ──
df = pd.read_parquet('../data/processed/features.parquet')
print(f"Loaded {len(df):,} rows, {df['ticker'].nunique()} tickers")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Columns: {list(df.columns)}")

# ── Backtest parameters ──
INITIAL_CAPITAL   = 100_000
COST_BPS          = 10          # 10 bps round-trip
MAX_HOLD_DAYS     = 20
MAX_POSITIONS      = 5          # max concurrent positions
POSITION_SIZE_PCT  = 0.20       # 20% of capital per position (1/5)

# ── Train / Test split ──
TRAIN_END = '2021-12-31'
TEST_START = '2022-01-01'

print(f"\nTrain: start → {TRAIN_END}")
print(f"Test:  {TEST_START} → end")
print(f"Capital: ${INITIAL_CAPITAL:,}, Cost: {COST_BPS}bps, Max positions: {MAX_POSITIONS}")

Loaded 823,610 rows, 472 tickers
Date range: 2018-01-02 → 2024-12-30
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'ticker', 'is_etf', 'ema8', 'ema21', 'ema50', 'avg_vol_10d', 'avg_vol_20d', 'daily_chg_pct', 'dollar_volume', 'prev_close', 'prev_ema8', 'prev_ema21', 'prev_ema50', 'prev_avg_vol_10d', 'prev_avg_vol_20d', 'prev_daily_chg_pct', 'prev_dollar_volume', 'screener_pass', 'sector_etf', 'regime_bull', 'sector_hot', 'vol_ratio', 'pct_from_ema8', 'ema_aligned', 'tight_consol', 'tight_consol_3pct', 'vol_declining_5d', 'vol_declining_2d', 'vol_spike', 'vol_spike_1_3', 'regime', 'sector_top3', 'entry_signal', 'entry_signal_v2', 'regime_score', 'entry_regime2', 'entry_v2_regime2', 'entry_regime3', 'entry_v2_regime3', 'entry_regime4', 'entry_v2_regime4', 'exit_trigger', 'next_open']

Train: start → 2021-12-31
Test:  2022-01-01 → end
Capital: $100,000, Cost: 10bps, Max positions: 5


In [2]:
import os
print(os.listdir('../data/processed/'))

['walk_forward_results.csv', 'backtest_trades.parquet', 'backtest_equity.parquet', 'optuna_best_params.json', 'clean.parquet', 'backtest_comparison.csv', 'features.parquet', 'final_comparison.csv', 'sector_map.csv']


In [3]:
# ============================================================
# CELL 3 — Run All Backtest Configurations & Save Results
# ============================================================
import sys, os
sys.path.insert(0, '../src')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from backtest import run_backtest, calc_metrics
from report_utils import set_plot_style, plot_equity_curve, plot_drawdown, plot_trade_distribution
import yfinance as yf

set_plot_style()
FIG_DIR = '../reports/figures/'
os.makedirs(FIG_DIR, exist_ok=True)

print('Downloading SPY...')
spy_raw = yf.download('SPY', start='2018-01-01', end='2025-01-01', progress=False)
spy_raw.index = pd.to_datetime(spy_raw.index)
spy_close = spy_raw[('Close', 'SPY')] if isinstance(spy_raw.columns, pd.MultiIndex) else spy_raw['Close']
spy_returns = spy_close.pct_change().dropna()
print(f'SPY loaded: {len(spy_returns)} days')

configs = [
    ('original',          'entry_signal',      None),
    ('original+SPY',      'entry_signal',      spy_returns),
    ('relaxed_v2',        'entry_signal_v2',   None),
    ('relaxed_v2+SPY',    'entry_signal_v2',   spy_returns),
    ('v2_regime>=3',      'entry_v2_regime3',  None),
    ('v2_regime>=3+SPY',  'entry_v2_regime3',  spy_returns),
    ('v2_regime>=4',      'entry_v2_regime4',  None),
    ('v2_regime>=4+SPY',  'entry_v2_regime4',  spy_returns),
]

df_sorted = df.sort_values(['ticker', 'date'])
all_metrics, equity_curves, primary_trades = [], {}, None

for name, sig_col, spy_ret in configs:
    if sig_col not in df_sorted.columns:
        print(f'  SKIP {name}: missing column {sig_col!r}'); continue
    print(f'Running {name}...', end=' ', flush=True)
    trades, equity = run_backtest(
        df_sorted, signal_col=sig_col,
        initial_capital=INITIAL_CAPITAL, cost_bps=COST_BPS,
        max_hold_days=MAX_HOLD_DAYS, max_positions=MAX_POSITIONS,
        position_size_pct=POSITION_SIZE_PCT, spy_returns=spy_ret,
    )
    if trades is None or len(trades) == 0:
        print('no trades'); continue
    equity_curves[name] = equity
    if name == 'relaxed_v2':
        primary_trades = trades
    for period in ['train', 'val', 'test']:
        t_p = trades[trades['period'] == period]
        if period == 'train':
            e_p = equity[equity['date'] <= pd.Timestamp(TRAIN_END)]
        elif period == 'val':
            e_p = equity[(equity['date'] > pd.Timestamp(TRAIN_END)) &
                         (equity['date'] < pd.Timestamp(TEST_START))]
        else:
            e_p = equity[equity['date'] >= pd.Timestamp(TEST_START)]
        m = calc_metrics(t_p, e_p.reset_index(drop=True), label=period, signal=name)
        all_metrics.append(m)
    print(f'{len(trades)} trades')

# Save comparison CSV
comp_df = pd.DataFrame(all_metrics)
comp_df.to_csv('../data/processed/backtest_comparison.csv', index=False)
print(f'\nSaved: backtest_comparison.csv ({len(comp_df)} rows)')

if 'relaxed_v2' in equity_curves:
    equity_curves['relaxed_v2'].to_parquet('../data/processed/backtest_equity.parquet', index=False)
if primary_trades is not None:
    primary_trades.to_parquet('../data/processed/backtest_trades.parquet', index=False)
    print('Saved: backtest_equity.parquet, backtest_trades.parquet')

# Equity curve figure
plot_equity_curve(
    {k: v for k, v in equity_curves.items() if '+SPY' not in k},
    splits=[
        (pd.Timestamp(TRAIN_END), 'Train->Val', '#888888'),
        (pd.Timestamp(TEST_START), 'Val->Test', '#D6604D'),
    ],
    title='DriftFire — Equity Curves',
    save_path=FIG_DIR + 'equity_curve.png',
)
plt.close('all')

if 'relaxed_v2' in equity_curves:
    plot_drawdown(equity_curves['relaxed_v2'],
                  title='DriftFire relaxed_v2 — Drawdown',
                  save_path=FIG_DIR + 'drawdown.png')
    plt.close('all')

if primary_trades is not None:
    plot_trade_distribution(primary_trades, save_path=FIG_DIR + 'trade_returns_dist.png')
    plt.close('all')

# Test period summary
test_df = comp_df[comp_df['period'] == 'test'][['signal','n_trades','sharpe','cagr_%','max_dd_%','hit_rate_%']]
print(f"\n{'='*70}")
print('TEST PERIOD SUMMARY')
print(f"{'='*70}")
print(test_df.to_string(index=False))


SPY loaded: 1760 days
Running original... 

19 trades
Running original+SPY... 

19 trades
Running relaxed_v2... 

412 trades
Running relaxed_v2+SPY... 

412 trades
Running v2_regime>=3... 

399 trades
Running v2_regime>=3+SPY... 

399 trades
Running v2_regime>=4... 

344 trades
Running v2_regime>=4+SPY... 

344 trades

Saved: backtest_comparison.csv (24 rows)
Saved: backtest_equity.parquet, backtest_trades.parquet



TEST PERIOD SUMMARY
          signal  n_trades  sharpe  cagr_%  max_dd_%  hit_rate_%
        original         4    0.20     1.3     -19.2        75.0
    original+SPY         4    0.21     0.7     -21.3        75.0
      relaxed_v2       164    0.98     2.4     -55.6        40.2
  relaxed_v2+SPY       164    0.98     1.9     -55.5        40.2
    v2_regime>=3       151    1.00     1.0     -58.1        36.4
v2_regime>=3+SPY       151    1.00     0.6     -58.1        36.4
    v2_regime>=4       131    0.95    -6.4     -61.0        37.4
v2_regime>=4+SPY       131    0.95    -6.9     -61.0        37.4
